# 影子策略简化回测 - Notebook版本

**重要说明**：
- 此notebook用于快速验证策略逻辑
- 数据来源：2024年实测数据（来自可信度总表）
- **不是10年完整回测结果**，需要在Ricequant平台实际运行

回测范围：2024年（验证可信度总表数据）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("数据来源说明：")
print("主线策略：假弱高开 +2.89%，胜率88.5%（2024年实测）")
print("观察线策略：胜率87.95%，盈亏比21.91（2024年实测）")
print("\n这些数据来自可信度总表_20260330.md的2024年实测结果")
print("\n⚠️ 下面的代码需要在Ricequant Notebook平台实际运行才能得到回测结果")

## 1. 快速验证：主线策略（2024年样本）

In [ ]:
# 主线策略参数
MAINLINE_PARAMS = {
    'market_cap_range': (50, 150),  # 亿元
    'position_limit': 30,  # % 相对历史最高
    'emotion_threshold': 30,  # 涨停家数
    'open_change_range': (0.001, 0.03),  # 开盘涨幅
    'sell_profit': 0.03,  # 冲高止盈
}

print("主线策略逻辑：")
print("1. 情绪过滤：涨停家数 >= 30")
print(f"2. 市值：{MAINLINE_PARAMS['market_cap_range'][0]}-{MAINLINE_PARAMS['market_cap_range'][1]}亿")
print(f"3. 位置：<= {MAINLINE_PARAMS['position_limit']}%")
print(f"4. 开盘涨幅：{MAINLINE_PARAMS['open_change_range'][0]*100:.1f}%-{MAINLINE_PARAMS['open_change_range'][1]*100:.1f}%")
print(f"5. 卖出：冲高+{MAINLINE_PARAMS['sell_profit']*100:.0f}%止盈或尾盘卖")
print("6. 停手：连亏3笔停3天")

In [ ]:
# 2024年实测数据（来自可信度总表）
mainline_2024_results = {
    'fake_weak_high_open': {
        'return': 0.0289,  # +2.89%
        'win_rate': 0.885,  # 88.5%
        'samples': 26,
    },
    'true_low_open_a': {
        'return': 0.0513,  # +5.13%
        'win_rate': 1.0,  # 100%
        'samples': 4,
    },
    'total_signals': 136,
    'daily_signals': 0.54,
}

print("\n2024年主线策略实测结果：")
print(f"假弱高开：收益 +{mainline_2024_results['fake_weak_high_open']['return']*100:.2f}%")
print(f"         胜率 {mainline_2024_results['fake_weak_high_open']['win_rate']*100:.1f}%")
print(f"         样本 {mainline_2024_results['fake_weak_high_open']['samples']}个")
print(f"\n真低开A：收益 +{mainline_2024_results['true_low_open_a']['return']*100:.2f}%")
print(f"         胜率 {mainline_2024_results['true_low_open_a']['win_rate']*100:.0f}%")
print(f"         样本 {mainline_2024_results['true_low_open_a']['samples']}个（极端样本）")
print(f"\n全年信号：{mainline_2024_results['total_signals']}个")
print(f"每日平均：{mainline_2024_results['daily_signals']}个")

## 2. 快速验证：观察线策略（2024年样本）

In [ ]:
# 观察线策略参数
OBSERVATION_PARAMS = {
    'board_count': 2,  # 只做二板
    'exclude_emotion': True,  # 不接情绪层
}

print("观察线策略逻辑：")
print(f"1. 只做二板（连板={OBSERVATION_PARAMS['board_count']}）")
print("2. 不做三板、四板")
print("3. 不做涨停排板")
print(f"4. 情绪层：{'关闭' if OBSERVATION_PARAMS['exclude_emotion'] else '开启'}")

In [ ]:
# 2024年实测数据（来自可信度总表）
observation_2024_results = {
    'win_rate': 0.8795,  # 87.95%
    'profit_loss_ratio': 21.91,
    'max_drawdown': 0.0060,  # 0.60%
}

print("\n2024年观察线策略实测结果：")
print(f"胜率：{observation_2024_results['win_rate']*100:.2f}%")
print(f"盈亏比：{observation_2024_results['profit_loss_ratio']:.2f}")
print(f"回撤：{observation_2024_results['max_drawdown']*100:.2f}%")

## 3. 简化回测逻辑（Ricequant API）

In [ ]:
# ⚠️ 以下代码需要在Ricequant Notebook平台实际运行

# 获取交易日列表
def get_test_dates():
    # Ricequant API
    # dates = get_trading_dates('2024-01-01', '2024-12-31')
    # return dates
    return []

# 获取涨停家数
def get_limit_up_count(date):
    # Ricequant API
    # all_stocks = list(get_all_securities().index)
    # count = 0
    # for stock in all_stocks[:500]:
    #     prices = get_price(stock, end_date=date, frequency='1d', fields=['close'], count=2)
    #     if len(prices) >= 2:
    #         pct = (prices['close'].iloc[-1] - prices['close'].iloc[-2]) / prices['close'].iloc[-2]
    #         if pct >= 0.095:
    #             count += 1
    # return count
    return 0

print("⚠️ 需要在Ricequant平台运行才能获取实际数据")

## 4. 可视化对比（基于可信度总表数据）

In [ ]:
# 策略对比
comparison_data = {
    '主线（假弱高开）': [
        f"+{mainline_2024_results['fake_weak_high_open']['return']*100:.2f}%",
        f"{mainline_2024_results['fake_weak_high_open']['win_rate']*100:.1f}%",
        f"{mainline_2024_results['fake_weak_high_open']['samples']}",
        f"{mainline_2024_results['daily_signals']}",
        'A档（实测）',
        '信号偏少',
    ],
    '观察线（二板）': [
        '-',
        f"{observation_2024_results['win_rate']*100:.2f}%",
        '-',
        '-',
        'A档（实测）',
        '表现稳定',
    ],
}

comparison_df = pd.DataFrame(comparison_data, 
                             index=['日内收益', '胜率', '样本数', '每日信号', '可信度', '状态'])

print("\n2024年策略对比：")
print(comparison_df)

In [ ]:
# 绘制胜率对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 主线胜率
axes[0].bar(['假弱高开', '真低开A'], 
            [mainline_2024_results['fake_weak_high_open']['win_rate']*100,
             mainline_2024_results['true_low_open_a']['win_rate']*100],
            color=['#2ecc71', '#3498db'])
axes[0].set_title('主线策略胜率（2024实测）', fontsize=14)
axes[0].set_ylabel('胜率 (%)', fontsize=12)
axes[0].set_ylim(0, 110)
axes[0].grid(True, alpha=0.3)

# 观察线胜率
axes[1].bar(['观察线（二板）'], 
            [observation_2024_results['win_rate']*100],
            color='#e74c3c')
axes[1].set_title('观察线策略胜率（2024实测）', fontsize=14)
axes[1].set_ylabel('胜率 (%)', fontsize=12)
axes[1].set_ylim(0, 100)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 完整10年回测方法

In [ ]:
print("\n完整10年回测方法：")
print("\n方法1：回测策略编辑器（如果180分钟未满）")
print("  - 上传 ricequant_strategy_editor.py 到策略编辑器")
print("  - 设置时间范围：2014-01-01 至 2024-12-31")
print("  - 运行回测（约需30-60分钟）")
print("  - 查看绩效报告")

print("\n方法2：Notebook完整版本")
print("  - 上传 backtest_10years.ipynb 到Notebook平台")
print("  - 运行所有单元格")
print("  - 查看净值曲线和绩效指标")

print("\n⚠️ 当前显示的数据是2024年实测结果，不是10年回测结果！")

## 6. 状态总结

In [ ]:
print("\n" + "="*80)
print("状态总结")
print("="*80)

print("\n主线策略（假弱高开）：")
print("  ✅ 2024年已实测：+2.89%收益，88.5%胜率")
print("  ⚠️ 信号偏少：每日平均0.54个")
print("  ❌ 10年回测：需要在平台实际运行")
print("  建议：极小仓影子盘验证")

print("\n观察线策略（二板）：")
print("  ✅ 2024年已实测：87.95%胜率，21.91盈亏比")
print("  ⚠️ 跨周期稳定性：2021-2023未实测")
print("  ❌ 10年回测：需要在平台实际运行")
print("  建议：补充历史验证，逐步加仓")

print("\n下一步操作：")
print("  1. 检查Ricequant回测编辑器时间（是否用满180分钟）")
print("  2. 上传ricequant_strategy_editor.py运行完整回测")
print("  3. 或在Notebook运行backtest_10years.ipynb")
print("  4. 查看实际10年回测结果")